In [3]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies.shape

(9742, 3)

In [5]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='str')

In [6]:
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
movies["description"] = (
    movies["title"] + " " + movies["genres"]
)

movies.head()

,movieId,title,genres,description
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure Children Fantasy,Jumanji (1995) Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men (1995) Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance,Waiting to Exhale (1995) Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["description"]
)

tfidf_matrix.shape

(9742, 9060)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    tfidf_matrix
)

similarity.shape

(9742, 9742)

In [10]:
def recommend_movie(movie_name, num_movies=10):

    movie_index = movies[
        movies["title"] == movie_name
    ].index[0]

    similarity_scores = list(
        enumerate(similarity[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    movie_indices = [
        i[0] for i in similarity_scores[1:num_movies+1]
    ]

    return movies.iloc[
        movie_indices
    ][
        ["title","genres"]
    ]

In [11]:
recommend_movie("Toy Story (1995)")

,title,genres
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy
7355,Toy Story 3 (2010),Adventure Animation Children Comedy Fantasy IMAX
3595,"Toy, The (1982)",Comedy
2539,We're Back! A Dinosaur's Story (1993),Adventure Animation Children Fantasy
26,Now and Then (1995),Children Drama
4089,Toy Soldiers (1991),Action Drama
1617,"NeverEnding Story, The (1984)",Adventure Children Fantasy
6194,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
12,Balto (1995),Adventure Animation Children


In [12]:
def search_movie(movie_name):

    results = movies[
        movies["title"]
        .str.contains(
            movie_name,
            case=False,
            regex=False
        )
    ]

    return results[["title","genres"]].head(10)

In [13]:
search_movie("toy story")

,title,genres
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy
7355,Toy Story 3 (2010),Adventure Animation Children Comedy Fantasy IMAX


In [14]:
def recommend_movie(movie_name, num_movies=10):

    matches = movies[
        movies["title"]
        .str.contains(
            movie_name,
            case=False,
            regex=False
        )
    ]

    if len(matches)==0:
        return "Movie not found"


    movie_index = matches.index[0]


    similarity_scores = list(
        enumerate(
            similarity[movie_index]
        )
    )


    similarity_scores = sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )


    movie_indices = [
        i[0]
        for i in similarity_scores[1:num_movies+1]
    ]


    return movies.iloc[
        movie_indices
    ][
        ["title","genres"]
    ]

In [15]:
recommend_movie("avatar")

,title,genres
7119,9 (2009),Adventure Animation Sci-Fi
7018,Star Trek (2009),Action Adventure Sci-Fi IMAX
8178,After Earth (2013),Action Adventure Sci-Fi IMAX
6797,Watchmen (2009),Action Drama Mystery Sci-Fi Thriller IMAX
7185,2012 (2009),Action Drama Sci-Fi Thriller
8426,Godzilla (2014),Action Adventure Sci-Fi IMAX
7693,"Avengers, The (2012)",Action Adventure Sci-Fi IMAX
3294,More (1998),Animation Drama Sci-Fi IMAX
7064,Transformers: Revenge of the Fallen (2009),Action Adventure Sci-Fi IMAX
8137,Oblivion (2013),Action Adventure Sci-Fi IMAX


In [16]:
import pickle


with open(
    "../models/similarity.pkl",
    "wb"
) as f:
    pickle.dump(similarity,f)


with open(
    "../models/movies.pkl",
    "wb"
) as f:
    pickle.dump(movies,f)


print("Model saved successfully")

Model saved successfully


# AI Semantic Recommendation using Embeddings

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
movie_embeddings = model.encode(
    movies["description"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

In [20]:
movie_embeddings.shape

(9742, 384)

In [21]:
import faiss

In [22]:
dimension = movie_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    movie_embeddings
)

print("FAISS index created")

FAISS index created


In [23]:
def semantic_recommend(query, n=10):

    query_vector = model.encode(
        [query]
    )


    distances, indices = index.search(
        query_vector,
        n
    )


    return movies.iloc[
        indices[0]
    ][
        ["title","genres"]
    ]

In [24]:
semantic_recommend(
    "space adventure with astronauts and survival"
)

,title,genres
9432,The Space Between Us (2016),Adventure Sci-Fi
3521,SpaceCamp (1986),Adventure Sci-Fi
1346,Lost in Space (1998),Action Adventure Sci-Fi
6791,Journey to the Center of the Earth (2008),Action Adventure Sci-Fi
4274,Journey to the Center of the Earth (1959),Adventure Children Sci-Fi
5004,Explorers (1985),Adventure Children Sci-Fi
8659,Space Buddies (2009),Adventure Children Fantasy Sci-Fi
6407,"Astronaut Farmer, The (2007)",Drama
8773,The Forgotten Space (2010),Documentary
9016,Doctor Who: The Waters of Mars (2009),Adventure Children Sci-Fi


In [25]:
import pickle


faiss.write_index(
    index,
    "../models/movie_embeddings.index"
)


with open(
    "../models/movie_embeddings.pkl",
    "wb"
) as f:
    pickle.dump(movie_embeddings,f)


print("AI model saved")

AI model saved


# Collaborative Filtering

In [26]:
import pandas as pd

ratings = pd.read_csv(
    "../data/ratings.csv"
)

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [27]:
ratings.shape

(100836, 4)

In [28]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

user_movie_matrix = user_movie_matrix.fillna(0)


movie_similarity = cosine_similarity(
    user_movie_matrix.T
)


movie_similarity.shape

(9724, 9724)

In [30]:
import pickle


with open(
    "../models/movie_similarity_rating.pkl",
    "wb"
) as f:
    pickle.dump(
        movie_similarity,
        f
    )


print("Saved")

Saved


In [31]:
movie_id_to_index = pd.Series(
    movies.index,
    index=movies.movieId
)

In [32]:
def hybrid_recommend(
    query,
    n=10,
    semantic_weight=0.6
):

    # Semantic search
    query_vector = model.encode(
        [query]
    )

    distances, indices = index.search(
        query_vector,
        50
    )


    semantic_scores = {}

    for i, distance in zip(
        indices[0],
        distances[0]
    ):
        semantic_scores[i] = 1/(1+distance)



    # Combine scores

    final_scores = {}


    for movie_index, score in semantic_scores.items():

        rating_score = 0


        if movie_index < movie_similarity.shape[0]:

            rating_score = movie_similarity[
                movie_index
            ].mean()


        final_scores[movie_index] = (
            semantic_weight * score
            +
            (1-semantic_weight) * rating_score
        )


    ranked_movies = sorted(
        final_scores.items(),
        key=lambda x:x[1],
        reverse=True
    )


    movie_indices = [
        x[0]
        for x in ranked_movies[:n]
    ]


    return movies.iloc[
        movie_indices
    ][
        ["title","genres"]
    ]

In [33]:
hybrid_recommend(
    "dark crime detective thriller"
)

,title,genres
4663,Darkman (1990),Action Crime Fantasy Sci-Fi Thriller
4242,Dark Blue (2003),Action Crime Drama Thriller
1945,Following (1998),Crime Mystery Thriller
6315,"Departed, The (2006)",Crime Drama Thriller
2912,Under Suspicion (2000),Crime Thriller
6590,Sleuth (2007),Drama Mystery Thriller
3778,High Crimes (2002),Thriller
7173,Saw VI (2009),Crime Horror Mystery Thriller
8545,Nightcrawler (2014),Crime Drama Thriller
7719,In Time (2011),Crime Sci-Fi Thriller


In [34]:
import pickle


with open(
    "../models/hybrid_ready.pkl",
    "wb"
) as f:
    pickle.dump(
        movie_similarity,
        f
    )

print("Hybrid model saved")

Hybrid model saved


In [35]:
import pickle
import numpy as np

top_k = 50

compressed_similarity = {}

for movie_id in range(movie_similarity.shape[0]):

    scores = movie_similarity[movie_id]

    top_movies = np.argsort(scores)[-top_k-1:-1]

    compressed_similarity[movie_id] = [
        (
            int(movie),
            float(scores[movie])
        )
        for movie in top_movies
    ]

print("Compression completed")

Compression completed


In [36]:
with open(
    "../models/movie_similarity_top50.pkl",
    "wb"
) as f:

    pickle.dump(
        compressed_similarity,
        f
    )

print("Saved successfully")

Saved successfully
